> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

In [ ]:
import sys
sys.path.insert(0, '../')

In [ ]:
import numpy as np
from scipy import stats
from utils import read_eeg, balance_labels, make_features

# Data loading

In [ ]:
X_train, y_train = read_eeg(why="train")
print("Before balancing:")
print(np.unique(y_train, return_counts=True))
print("")
X_train, y_train = balance_labels(X_train, y_train)
print("After balancing:")
print(np.unique(y_train, return_counts=True))


In [ ]:
X_test, y_test = read_eeg(why="test", who=7, verbose=True)
print("Before balancing: ")
print(np.unique(y_test, return_counts=True))
print("")

In [ ]:
X_train = make_features(X_train)
X_test = make_features(X_test)

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# 1. Initialize the model
model =  LogisticRegression()
# 2. Fit on training data
model.fit(X_train, y_train)

# 3. Predict on test data
y_pred = model.predict(X_test)
# 4. Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
accuracy = accuracy_score(y_test, np.round(y_pred))

print("Mean Squared Error:", mse)
print("R² Score:", r2)
print("Accuracy:", accuracy*100, "%")

In [ ]:
# --- cell 1: coloured time-line plot ---------------------------------------
import numpy as np
import matplotlib.pyplot as plt

# y_true : 1-D array of true labels   (shape n_samples,)
# y_pred : 1-D array of predictions   (same length)

# --- pick three colours for the three classes ---
colour_map = {0: 'tab:red', 1: 'tab:green', 2: 'tab:blue'}

plt.figure(figsize=(14, 4))

# scatter each class in its own colour
for cls in np.unique(y_test):
    idx = np.where(y_test == cls)[0]
    plt.scatter(idx, y_test[idx],
                color=colour_map[cls],
                s=12, marker='o', label=f'True class {cls}')

# overlay the regression output as a thin black line
plt.plot(y_pred, color='k', lw=1.2, label='Predicted value')

plt.xlabel('Sample index')
plt.ylabel('Class / prediction')
plt.title('True classes (coloured) + Linear-Regression output')
plt.legend(ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
# --- cell 2: confusion-matrix ----------------------------------------------
import seaborn as sns
from sklearn.metrics import confusion_matrix

y_pred_round = np.round(y_pred).astype(int)      # convert regression output to class labels
cm = confusion_matrix(y_test, y_pred_round, labels=[0, 1, 2])

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[0,1,2], yticklabels=[0,1,2])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion matrix')
plt.tight_layout()
plt.show()

In [ ]:
best_acc = 0
best_i = -1

for i in range(26):
    X_test, y_test = read_eeg(why="test", who=i)
    X_test = make_features(X_test)
    # 1. Initialize the model
    model =  LogisticRegression()
    # 2. Fit on training data
    model.fit(X_train, y_train)

    # 3. Predict on test data
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, np.round(y_pred))
    if accuracy > best_acc:
        best_acc = accuracy
        best_i = i
        print("Best accuracy: ", best_acc)
        print("Best i: ", best_i)